# ChromaDB — Keyword Search & Metadata Filtering

Query the collection using **keyword / full-text search** and **metadata filters**.

Available metadata fields per chunk:

| Field | Example values |
|---|---|
| `title` | `"Gamage et al. - 2025 - ..."` |
| `format` | `"pdf"`, `"ipynb"`, `"md"` |
| `source_corpus` | `"paper"`, `"mlflow"` |
| `content_type` | `"narrative"`, `"code"` |
| `source_file` | full relative path |
| `page` | integer (PDFs) |
| `cell_index` | integer (notebooks) |

In [1]:
from pathlib import Path
import sys
import json

import chromadb
import pandas as pd
from chromadb.api.types import Documents, EmbeddingFunction, Embeddings

# Resolve repo root so app.config is importable regardless of cwd
repo_root = next(
    (p for p in [Path.cwd(), *Path.cwd().resolve().parents] if (p / "pyproject.toml").exists()),
    Path.cwd().resolve(),
)
sys.path.insert(0, str(repo_root))

from app.config import (
    CHROMA_HOST,
    CHROMA_PORT,
    CHROMA_SSL,
    COLLECTION_NAME,
    EMBEDDING_PROVIDER,
    EMBEDDING_MODEL,
)
from app.factory import get_embeddings

client = chromadb.HttpClient(host=CHROMA_HOST, port=CHROMA_PORT, ssl=CHROMA_SSL)

# Use the exact same embedding backend/model as ingestion to avoid dimension mismatch.
lc_embeddings = get_embeddings()

class LangChainEmbeddingFunction(EmbeddingFunction[Documents]):
    def __init__(self, langchain_embeddings) -> None:
        self._embeddings = langchain_embeddings

    def __call__(self, input: Documents) -> Embeddings:
        return self._embeddings.embed_documents(list(input))

collection = client.get_collection(
    COLLECTION_NAME,
    embedding_function=LangChainEmbeddingFunction(lc_embeddings),
)

print(f"Collection : {COLLECTION_NAME}")
print(f"Embedding  : {EMBEDDING_PROVIDER}/{EMBEDDING_MODEL}")
print(f"Total docs : {collection.count()}")

/home/mahee/Work/Thesis/Repos/langchain-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Collection : nomic_embed_text
Embedding  : ollama/nomic-embed-text
Total docs : 105


---
## 1. Keyword / substring search

ChromaDB supports `$contains` and `$not_contains` on the **document text** via `where_document`.

In [2]:
KEYWORD = "NISQ"   # ← change me
N       = 10                     # max results

results = collection.query(
    query_texts=[KEYWORD],        # ChromaDB embeds this for you
    n_results=N,
    where_document={"$contains": KEYWORD},
    include=["documents", "metadatas", "distances"],
)

rows = [
    {
        "distance": round(dist, 4),
        "title":    meta.get("title", "")[:60],
        "format":   meta.get("format", ""),
        "corpus":   meta.get("source_corpus", ""),
        "type":     meta.get("content_type", ""),
        "snippet":  doc[:120].replace("\n", " "),
    }
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    )
]

pd.set_option("display.max_colwidth", None)
pd.DataFrame(rows)

,distance,title,format,corpus,type,snippet
0,0.9695,Preskill - 2018 - Quantum Computing in the NISQ era and beyo,md,unknown,narrative,## Quantum Computing in the NISQ era and beyond ## John Preskill Institute for Quantum Information and Matter and Wa
1,0.9761,Preskill - 2018 - Quantum Computing in the NISQ era and beyo,md,unknown,narrative,Let me now summarize the main points I’ve tried to convey. - Now is a privileged time in the history of science and te
2,1.0254,Preskill - 2018 - Quantum Computing in the NISQ era and beyo,md,unknown,narrative,"- NISQ is not likely to change the world all by itself. Instead, the primary goal for nearterm quantum platforms should"
3,1.0344,Gamage et al. - 2025 - Enhancing Quantum Software Developmen,md,unknown,narrative,2025 IEEE International Conference on Quantum Computing and Engineering (QCE) # Enhancing Quantum Software Development
4,1.0448,Preskill - 2018 - Quantum Computing in the NISQ era and beyo,md,unknown,narrative,"Physicists are excited about this NISQ technology, which gives us new tools for exploring the physics of many entangled"
5,1.0645,Weder et al. - 2021 - QProv A provenance system for quantum,md,qprov,narrative,"12. Maciejewski, F.B., et al.: Mitigation of readout noise in near‐term quantum devices by classical post‐processing bas"
6,1.0717,Preskill - 2018 - Quantum Computing in the NISQ era and beyo,md,unknown,narrative,"There is a substantial opportunity for experimentalists and theorists, working together over the next few years, to find"
7,1.1357,Gamage et al. - 2025 - Enhancing Quantum Software Developmen,md,unknown,narrative,"- [3] J. Preskill. Quantum Computing in the NISQ era and beyond. _Quantum_ , 2:79, Aug. 2018. - [4] P. Senapati, Z. Wa"
8,1.1379,Preskill - 2018 - Quantum Computing in the NISQ era and beyo,md,unknown,narrative,A few years ago I spoke enthusiastically about _quantum supremacy_ as an impending milestone for human civilization [22]
9,1.1448,Preskill - 2018 - Quantum Computing in the NISQ era and beyo,md,unknown,narrative,## 4.2 Qubit “quality” I’ve emphasized the number of qubits as a measure of how difficult it is to do the simulation o


---
## 2. Filter by metadata

Use `where` with ChromaDB's filter operators:
`$eq`, `$ne`, `$in`, `$nin`, `$and`, `$or`

Filters can be combined with keyword search or used standalone via `collection.get()`.

In [6]:
# ── Filter by format ─────────────────────────────────────────────────────────
# Change FORMAT to "pdf", "ipynb", "md", etc.
FORMAT = "pdf"

results = collection.get(
    where={"format": {"$eq": FORMAT}},
    limit=10,
    include=["documents", "metadatas"],
)

rows = [
    {
        "title":  meta.get("title", "")[:70],
        "format": meta.get("format", ""),
        "corpus": meta.get("source_corpus", ""),
        "page":   meta.get("page", ""),
        "snippet": doc[:100].replace("\n", " "),
    }
    for doc, meta in zip(results["documents"], results["metadatas"])
]
pd.DataFrame(rows)

,title,format,corpus,page,snippet
0,Gamage et al. - 2025 - Enhancing Quantum Software Development Process,pdf,paper,0,Enhancing Quantum Software Development Process with Experiment Tracking Mahee Gamage University of J
1,Gamage et al. - 2025 - Enhancing Quantum Software Development Process,pdf,paper,0,this complicated environment in a structured way following a well-defined software development lifec
2,Gamage et al. - 2025 - Enhancing Quantum Software Development Process,pdf,paper,0,"experiment tracking, reproducibility, and production deploy- ment. We can observe that quantum softw"
3,Gamage et al. - 2025 - Enhancing Quantum Software Development Process,pdf,paper,1,User organization VTT 4. view 1. execute program 2. get results and calibration data Qx Service3. tr
4,Gamage et al. - 2025 - Enhancing Quantum Software Development Process,pdf,paper,1,"teams, as depicted in Fig. 1. III. Q UANTUM EXPERIMENT TRACKING Since its introduction, MLflow4 has"
5,Gamage et al. - 2025 - Enhancing Quantum Software Development Process,pdf,paper,1,"that MLflow is well-suited to quantum research, support- ing improved development practices, reprodu"
6,Gamage et al. - 2025 - Enhancing Quantum Software Development Process,pdf,paper,1,"Association for Computing Machinery. [6] B. Weder, J. Barzen, F. Leymann, M. Salm, and K. Wild. Qpro"
7,Kinanen et al. - 2025 - Toolchain for Experiment Tracking in Iterative,pdf,paper,0,Toolchain for experiment tracking in iterative quantum software development Otso Kinanen University
8,Kinanen et al. - 2025 - Toolchain for Experiment Tracking in Iterative,pdf,paper,0,"However, QSE diverges from classical software engineering due to the inherent uncertainty of quantum"
9,Kinanen et al. - 2025 - Toolchain for Experiment Tracking in Iterative,pdf,paper,0,"ground in Section II, followed by the motivation and objectives of our work in Section III. Next we"


In [7]:
# ── Filter by source_corpus ───────────────────────────────────────────────────
# e.g. "paper", "mlflow"
CORPUS = "paper"

results = collection.get(
    where={"source_corpus": {"$eq": CORPUS}},
    limit=10,
    include=["documents", "metadatas"],
)

rows = [
    {
        "title":  meta.get("title", "")[:70],
        "format": meta.get("format", ""),
        "page":   meta.get("page", ""),
        "snippet": doc[:100].replace("\n", " "),
    }
    for doc, meta in zip(results["documents"], results["metadatas"])
]
pd.DataFrame(rows)

,title,format,page,snippet
0,Gamage et al. - 2025 - Enhancing Quantum Software Development Process,pdf,0,Enhancing Quantum Software Development Process with Experiment Tracking Mahee Gamage University of J
1,Gamage et al. - 2025 - Enhancing Quantum Software Development Process,pdf,0,this complicated environment in a structured way following a well-defined software development lifec
2,Gamage et al. - 2025 - Enhancing Quantum Software Development Process,pdf,0,"experiment tracking, reproducibility, and production deploy- ment. We can observe that quantum softw"
3,Gamage et al. - 2025 - Enhancing Quantum Software Development Process,pdf,1,User organization VTT 4. view 1. execute program 2. get results and calibration data Qx Service3. tr
4,Gamage et al. - 2025 - Enhancing Quantum Software Development Process,pdf,1,"teams, as depicted in Fig. 1. III. Q UANTUM EXPERIMENT TRACKING Since its introduction, MLflow4 has"
5,Gamage et al. - 2025 - Enhancing Quantum Software Development Process,pdf,1,"that MLflow is well-suited to quantum research, support- ing improved development practices, reprodu"
6,Gamage et al. - 2025 - Enhancing Quantum Software Development Process,pdf,1,"Association for Computing Machinery. [6] B. Weder, J. Barzen, F. Leymann, M. Salm, and K. Wild. Qpro"
7,Kinanen et al. - 2025 - Toolchain for Experiment Tracking in Iterative,pdf,0,Toolchain for experiment tracking in iterative quantum software development Otso Kinanen University
8,Kinanen et al. - 2025 - Toolchain for Experiment Tracking in Iterative,pdf,0,"However, QSE diverges from classical software engineering due to the inherent uncertainty of quantum"
9,Kinanen et al. - 2025 - Toolchain for Experiment Tracking in Iterative,pdf,0,"ground in Section II, followed by the motivation and objectives of our work in Section III. Next we"


In [8]:
# ── Filter by title (exact match) ─────────────────────────────────────────────
TITLE = "Gamage et al. - 2025 - Enhancing Quantum Software Development Process with Experiment Tracking"

results = collection.get(
    where={"title": {"$eq": TITLE}},
    include=["documents", "metadatas"],
)

print(f"Chunks for this title: {len(results['documents'])}")
for i, (doc, meta) in enumerate(zip(results["documents"], results["metadatas"])):
    print(f"\n── Chunk {i} | page={meta.get('page', meta.get('cell_index', '?'))} ──")
    print(doc[:300])

Chunks for this title: 7

── Chunk 0 | page=0 ──
Enhancing Quantum Software Development Process
with Experiment Tracking
Mahee Gamage
University of Jyv¨askyl¨a
Jyv¨askyl¨a, Finland
mahee.s.hewagamage@jyu.fi
Otso Kinanen
University of Jyv¨askyl¨a
Jyv¨askyl¨a, Finland
otso.j.r.kinanen@jyu.fi
Jake Muff
Quantum Algorithms and Software
VTT Technical Re

── Chunk 1 | page=0 ──
this complicated environment in a structured way following
a well-defined software development lifecycle [5], supported
by specialized tools [1]. Given the limited availability of
quantum hardware, the development process begins on quan-
tum simulators. Subsequently, as the program or algorithm
matu

── Chunk 2 | page=0 ──
experiment tracking, reproducibility, and production deploy-
ment. We can observe that quantum software development
has some similarities with ML/AI development. For example,
developers have the option to use several quantum software
development toolkits (e.g. Qiskit1, PennyLane2, Qrisp3, etc.)
that


---
## 3. Combined: keyword search + metadata filter

Semantic query restricted to a specific format or corpus.

In [9]:
QUERY  = "experiment reproducibility"   # semantic query
FORMAT = "ipynb"                         # metadata filter
N      = 5

results = collection.query(
    query_texts=[QUERY],
    n_results=N,
    where={"format": {"$eq": FORMAT}},
    include=["documents", "metadatas", "distances"],
)

for doc, meta, dist in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0],
):
    print(f"dist={dist:.4f} | {meta.get('format')} | {meta.get('title', '')[:60]}")
    print(doc[:200].replace("\n", " "))
    print()

dist=0.6920 | ipynb | MLflow with Optuna: Hyperparameter Optimization and Tracking
#### Create an experiment for our hyperparameter tuning runs

dist=0.7134 | ipynb | MLflow with Optuna: Hyperparameter Optimization and Tracking
def get_or_create_experiment(experiment_name):     """     Retrieve the ID of an existing MLflow experiment or create a new one if it doesn't exist.      This function checks if an experiment with the

dist=0.7144 | ipynb | MLflow with Optuna: Hyperparameter Optimization and Tracking
experiment_id = get_or_create_experiment("Apples Demand")

dist=0.7314 | ipynb | MLflow with Optuna: Hyperparameter Optimization and Tracking
### Setting Up the MLflow Experiment  Before we start our hyperparameter tuning process, we need to designate a specific "experiment" within MLflow to track and log our results. An experiment in MLflo

dist=0.7328 | ipynb | Logging Visualizations with MLflow
# Set seed for reproducibility     np.random.seed(9999)      # Create date range     d

In [10]:
# ── $and: multiple metadata conditions ───────────────────────────────────────
QUERY   = "model tracking"
CORPUS  = "mlflow"
C_TYPE  = "code"
N       = 5

results = collection.query(
    query_texts=[QUERY],
    n_results=N,
    where={
        "$and": [
            {"source_corpus": {"$eq": CORPUS}},
            {"content_type":  {"$eq": C_TYPE}},
        ]
    },
    include=["documents", "metadatas", "distances"],
)

for doc, meta, dist in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0],
):
    print(f"dist={dist:.4f} | corpus={meta.get('source_corpus')} | type={meta.get('content_type')}")
    print(doc[:200].replace("\n", " "))
    print()

dist=0.6941 | corpus=mlflow | type=code
mlflow.set_tracking_uri("http://127.0.0.1:8080")  mlflow.set_experiment("Visualizations Demo")  X = my_data.drop(columns=["demand", "date"]) y = my_data["demand"] X_train, X_test, y_train, y_test = tr

dist=0.7662 | corpus=mlflow | type=code
model_output  # --- output --- 0  1  2  3  4   5   6   7   8   9 0  5  6  7  8  9  10  11  12  13  14

dist=0.7665 | corpus=mlflow | type=code
# If you are running this tutorial in local mode, leave the next line commented out. # Otherwise, uncomment the following line and set your tracking uri to your local or remote tracking server.  mlflo

dist=0.7702 | corpus=mlflow | type=code
# Load the saved model loaded_model = mlflow.pyfunc.load_model(model_path)

dist=0.7704 | corpus=mlflow | type=code
# If you are running this tutorial in local mode, leave the next line commented out. # Otherwise, uncomment the following line and set your tracking uri to your local or remote tracking server.  # mlf



---
## 4. Browse collection metadata

Inspect what titles, formats, and corpora are actually stored.

In [11]:
# Fetch all metadata (no documents) — useful for large collections
all_meta = collection.get(include=["metadatas"])["metadatas"]

df_meta = pd.DataFrame(all_meta)
print(f"Total chunks: {len(df_meta)}")
df_meta.head(10)

Total chunks: 907


,source_corpus,source_file,content_type,page,title,format,description,cell_index
0,paper,knowledge_ingestion/content/v2/content/orig_paper/Gamage et al. - 2025 - Enhancing Quantum Software Development Process with Experiment Tracking.pdf,narrative,0.0,Gamage et al. - 2025 - Enhancing Quantum Software Development Process with Experiment Tracking,pdf,NaN,NaN
1,paper,knowledge_ingestion/content/v2/content/orig_paper/Gamage et al. - 2025 - Enhancing Quantum Software Development Process with Experiment Tracking.pdf,narrative,0.0,Gamage et al. - 2025 - Enhancing Quantum Software Development Process with Experiment Tracking,pdf,NaN,NaN
2,paper,knowledge_ingestion/content/v2/content/orig_paper/Gamage et al. - 2025 - Enhancing Quantum Software Development Process with Experiment Tracking.pdf,narrative,0.0,Gamage et al. - 2025 - Enhancing Quantum Software Development Process with Experiment Tracking,pdf,NaN,NaN
3,paper,knowledge_ingestion/content/v2/content/orig_paper/Gamage et al. - 2025 - Enhancing Quantum Software Development Process with Experiment Tracking.pdf,narrative,1.0,Gamage et al. - 2025 - Enhancing Quantum Software Development Process with Experiment Tracking,pdf,NaN,NaN
4,paper,knowledge_ingestion/content/v2/content/orig_paper/Gamage et al. - 2025 - Enhancing Quantum Software Development Process with Experiment Tracking.pdf,narrative,1.0,Gamage et al. - 2025 - Enhancing Quantum Software Development Process with Experiment Tracking,pdf,NaN,NaN
5,paper,knowledge_ingestion/content/v2/content/orig_paper/Gamage et al. - 2025 - Enhancing Quantum Software Development Process with Experiment Tracking.pdf,narrative,1.0,Gamage et al. - 2025 - Enhancing Quantum Software Development Process with Experiment Tracking,pdf,NaN,NaN
6,paper,knowledge_ingestion/content/v2/content/orig_paper/Gamage et al. - 2025 - Enhancing Quantum Software Development Process with Experiment Tracking.pdf,narrative,1.0,Gamage et al. - 2025 - Enhancing Quantum Software Development Process with Experiment Tracking,pdf,NaN,NaN
7,paper,knowledge_ingestion/content/v2/content/orig_paper/Kinanen et al. - 2025 - Toolchain for Experiment Tracking in Iterative Quantum Software Development.pdf,narrative,0.0,Kinanen et al. - 2025 - Toolchain for Experiment Tracking in Iterative Quantum Software Development,pdf,NaN,NaN
8,paper,knowledge_ingestion/content/v2/content/orig_paper/Kinanen et al. - 2025 - Toolchain for Experiment Tracking in Iterative Quantum Software Development.pdf,narrative,0.0,Kinanen et al. - 2025 - Toolchain for Experiment Tracking in Iterative Quantum Software Development,pdf,NaN,NaN
9,paper,knowledge_ingestion/content/v2/content/orig_paper/Kinanen et al. - 2025 - Toolchain for Experiment Tracking in Iterative Quantum Software Development.pdf,narrative,0.0,Kinanen et al. - 2025 - Toolchain for Experiment Tracking in Iterative Quantum Software Development,pdf,NaN,NaN


In [12]:
# Unique titles and how many chunks each has
df_meta.groupby("title").size().reset_index(name="chunks").sort_values("chunks", ascending=False)

,title,chunks
27,index,244
21,MLflow with Optuna: Hyperparameter Optimization and Tracking,50
23,Preskill - 2018 - Quantum Computing in the NISQ era and beyond,47
6,Fine-Tuning Open-Source LLM using QLoRA with MLflow and PEFT,44
3,Customizing a Model's predict method,40
7,Fine-Tuning Transformers with MLflow for Enhanced Model Management,39
20,MLflow Signature Playground Notebook,37
18,Logging Visualizations with MLflow,34
15,Introduction to Translation with Transformers and MLflow,27
4,Deploy an MLflow `PyFunc` model with Model Serving,24


In [13]:
# Chunk distribution by format
df_meta["format"].value_counts().rename_axis("format").reset_index(name="chunks")

,format,chunks
0,ipynb,497
1,mdx,299
2,pdf,73
3,md,38


In [14]:
# Chunk distribution by source_corpus
df_meta["source_corpus"].value_counts().rename_axis("source_corpus").reset_index(name="chunks")

,source_corpus,chunks
0,mlflow,801
1,paper,73
2,qiskit,18
3,qprov,15


In [15]:
# Chunk distribution by content_type
df_meta["content_type"].value_counts().rename_axis("content_type").reset_index(name="chunks")

,content_type,chunks
0,narrative,667
1,code,240
